# v010_entity_blocking — Blocking sweep: name_addr_word top_k 25 → 35

| Field | Value |
|---|---|
| **Version** | `v010_entity_blocking` |
| **Plan group** | A2 (blocking sweep: top_k parameter) |
| **Parent version** | v001 |
| **Author** | M2 |
| **Date** | 2026-09-26 |
| **Status** | kept |

This is a **blocking-only experiment** (plan group A2). No model, features, or
decision logic changes. The sole change is raising `name_addr_word.top_k` from
25 to 35 in `BlockingConfig`, which allows the name+address word TF-IDF retrieval
pass to return up to 10 extra candidates per S1 entity.

**What is NOT in scope:** `local_f05` (matcher F0.5) is not reported here because
the matcher is frozen at v001. Only blocking metrics are evaluated.
The `cand_recall` column in `experiments.csv` is the **blocking pair recall**,
not a final model score.

### Sweep conducted (India partition, 10k-S1 sample) — prior results

| Config | Pair Recall | Ceiling F0.5 | Cands/S1 | Time |
|---|---:|---:|---:|---:|
| baseline (top_k=25) | 0.9636 | 0.9866 | 31.4 | 182s |
| max_df 0.01→0.02 | identical | — | — | — |
| top_k 25→35 | 0.9655 | 0.9873 | 35.5 | 197s |
| max_per_s1 60→80 | 0.9637 | 0.9866 | 31.5 | 250s |
| P4 addr-char ON | 0.9658 | 0.9876 | 36.3 | 468s |
| P4 ON + cap 80 | 0.9662 | 0.9878 | 36.5 | 470s |

**Selected config:** `top_k=35` — best cost/benefit among the sweep candidates.
Findings:
1. `max_df 0.01→0.02` is a dead knob; `max_df_abs=10_000` already binds tighter.
2. Raising `max_per_s1` alone does not materially improve recall.
3. P4 (address char-gram pass) costs ~2.6× runtime, exceeding the 10-minute budget on larger partitions.
4. `top_k 25→35` is the strongest cost/benefit candidate and is validated below on the FULL fold.


## 1. Hypothesis

* **Change vs parent (v001):** `name_addr_word.top_k` 25 → 35. All other blocking
  parameters (exact keys, `max_df_abs`, `max_per_s1`, pass structure) are identical.
* **Why it should improve recall:** The name+address word TF-IDF pass is the
  workhorse; raising top_k from 25 to 35 retrieves 10 more candidates per S1 entity,
  recovering true pairs that previously ranked 26th–35th. The India sweep showed
  +0.0019 pair recall at only +4.1 cands/S1 extra cost.
* **Expected effect:** pair recall +0.0015–0.0025 vs v001; cands/S1 +3–5;
  runtime ≤10 min on full fold.
* **Discard if:** blocking runtime > 10 min on the full fold, OR pair recall < 0.97,
  OR mean cands/S1 > 40.

## 2. Setup

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import psutil

from entity_resolution import config as C
from entity_resolution.blocking import BlockingConfig, TopKSpec, block, PASS_BITS
from entity_resolution.evaluate import blocking_report
from entity_resolution.pipeline import PipelineConfig, load_normalised, normalise_split, pool_of
from entity_resolution.split import load_fold, Fold
from entity_resolution.tracking import log_result, git_commit, timed

EXP_DIR = C.EXPERIMENTS / "v010_entity_blocking"
ARTIFACTS = EXP_DIR / "artifacts"
DATASET_DIR = C.DATASET
CACHE_DIR = DATASET_DIR / ".cache" / "pipeline"

# The one change: top_k 25 -> 35. All other parameters are v001 defaults.
TOP_K_35_CFG = BlockingConfig(
    name_addr_word=TopKSpec("name_addr", "word", (1, 2), 35, 0.20, 0.01, max_df_abs=10_000)
)
BASELINE_CFG = BlockingConfig()  # v001 top_k=25

PIPE_CFG = PipelineConfig(dataset_dir=DATASET_DIR, cache_dir=CACHE_DIR)

print(f"top_k=35 config key: {TOP_K_35_CFG.key()}")
print(f"baseline   config key: {BASELINE_CFG.key()}")
print(f"git commit: {git_commit()}")

## 3. Data — Load val fold + normalised frames

The fixed validation split is 20% of training Source 1 entities (by ID hash),
with their matched pool records. Normalised frames are loaded from the pipeline
Parquet cache (built once by v001). The learned token map from v001 is applied.

In [ ]:
# Load v001 token map
token_map_candidates = sorted(CACHE_DIR.glob("token_map_*.json"))
token_map = json.loads(token_map_candidates[0].read_text()) if token_map_candidates else {}
print(f"Token map: {len(token_map)} entries ({token_map_candidates[0].name if token_map_candidates else 'none'})")

normalise_split("train", PIPE_CFG)
print("Normalised caches: OK")

In [ ]:
t0 = time.perf_counter()
val_fold = load_fold("val", DATASET_DIR)
s1n = load_normalised("train", (1,), PIPE_CFG, val_fold.s1[C.ENTITY_ID], token_map)
pooln = load_normalised("train", (2, 3), PIPE_CFG,
                        pool_of(val_fold)[C.ENTITY_ID], token_map)
load_secs = time.perf_counter() - t0

pd.DataFrame([val_fold.summary()]).assign(
    countries=str(sorted(s1n[C.COUNTRY].unique())),
    load_seconds=round(load_secs, 1)
)

## 4. Method — Blocking with top_k=35

The `block()` function partitions by country, runs the three exact-key passes and
two TF-IDF top-k passes (name char-grams + name+address word bigrams), unions the
results, and applies the `max_per_s1=60` cap (exact pairs first, then by similarity).

Results are cached per country and config hash under the pipeline cache dir;
re-running the cell reads from disk if the parquet already exists.

In [ ]:
SWEEP_TAG = f"val_{len(s1n)}_{len(pooln)}_sweep"
print(f"Cache tag: {SWEEP_TAG}")
print(f"Cache dir: {CACHE_DIR / 'pairs'}")

t0 = time.perf_counter()
pairs_35 = block(s1n, pooln, TOP_K_35_CFG,
                 cache_dir=CACHE_DIR / "pairs", tag=SWEEP_TAG)
blocking_secs = time.perf_counter() - t0
peak_rss = psutil.Process().memory_info().rss / 1e9

print(f"Pairs:        {len(pairs_35):,}")
print(f"Blocking:     {blocking_secs:.1f}s ({blocking_secs/60:.2f} min)")
print(f"Peak RSS:     {peak_rss:.2f} GB")

## 5. Evaluation — Full blocking report

`blocking_report()` computes pair recall, entity recall, ceiling F0.5,
and candidate distribution statistics over the entire val fold.

In [ ]:
report_35 = blocking_report(pairs_35, val_fold)
pd.Series(report_35).rename("top_k=35")

In [ ]:
# Per-country recall
country_recalls = {}
rows = []
for country in sorted(s1n[C.COUNTRY].unique()):
    s1c = s1n[s1n[C.COUNTRY] == country].reset_index(drop=True)
    poolc = pooln[pooln[C.COUNTRY] == country].reset_index(drop=True)
    s1c_ids = set(s1c[C.ENTITY_ID])
    poolc_ids = set(poolc[C.ENTITY_ID])
    pairs_c = pairs_35[pairs_35[C.S1_ID].isin(s1c_ids)].reset_index(drop=True)
    truth_c = val_fold.pairs[val_fold.pairs[C.S1_ID].isin(s1c_ids)].reset_index(drop=True)
    s2c = val_fold.s2[val_fold.s2[C.ENTITY_ID].isin(poolc_ids)].reset_index(drop=True)
    s3c = val_fold.s3[val_fold.s3[C.ENTITY_ID].isin(poolc_ids)].reset_index(drop=True)
    if len(truth_c) == 0:
        continue
    fold_c = Fold(country, s1c, s2c, s3c, truth_c)
    r_c = blocking_report(pairs_c, fold_c)
    country_recalls[country] = r_c["pair_recall"]
    rows.append({"country": country, **{k: round(v, 4) if isinstance(v, float) else v
                                        for k, v in r_c.items()}})
pd.DataFrame(rows).set_index("country")

In [ ]:
# Load baseline from its cache
base_cache_dir = CACHE_DIR / "pairs" / "val_441333_83c0d952_2064065_d436c334_mcd072aee"
base_key = BASELINE_CFG.key()
base_parts = []
for country_dir in sorted(base_cache_dir.iterdir()):
    p = country_dir / f"pairs_{base_key}.parquet"
    if p.exists():
        base_parts.append(pd.read_parquet(p))
pairs_base = pd.concat(base_parts, ignore_index=True).sort_values(C.S1_ID, kind="stable").reset_index(drop=True)
report_base = blocking_report(pairs_base, val_fold)

# Comparison table
compare = pd.DataFrame({
    "baseline (top_k=25)": pd.Series(report_base),
    "top_k=35": pd.Series(report_35),
})
compare["delta"] = compare["top_k=35"] - compare["baseline (top_k=25)"]
compare.round(4)

In [ ]:
# M2 target check
pr = report_35["pair_recall"]
cm = report_35["candidates_mean"]
rt_min = blocking_secs / 60.0
rss = peak_rss

targets = pd.Series({
    "pair_recall >= 0.97": f"{pr:.4f}  {'PASS' if pr >= 0.97 else 'FAIL'}",
    "mean cands/S1 <= 40": f"{cm:.2f}  {'PASS' if cm <= 40 else 'FAIL'}",
    "runtime <= 10 min":   f"{rt_min:.2f} min  {'PASS' if rt_min <= 10 else 'FAIL'}",
    "peak RAM <= 3 GB":    f"{rss:.2f} GB  {'PASS' if rss <= 3 else 'FAIL'}",
})
targets

## 6. Log the result

In [ ]:
notes = (
    f"pair_recall={report_35['pair_recall']:.4f}; "
    f"entity_recall={report_35['entity_recall']:.4f}; "
    f"ceiling_f05={report_35['ceiling_f_beta']:.4f}; "
    f"cands_mean={report_35['candidates_mean']:.2f}/S1; "
    f"cands_p95={report_35['candidates_p95']:.0f}; "
    f"runtime={blocking_secs:.0f}s; "
    f"delta_recall={report_35['pair_recall'] - report_base['pair_recall']:+.4f} vs v001"
)

if pr >= 0.97 and cm <= 40 and rt_min <= 10:
    decision = "KEEP"
else:
    decision = "DROP"

row = log_result(
    EXP_DIR,
    change="blocking top_k 25->35 (name_addr_word pass)",
    group="A2",
    local_f05=None,   # blocking-only: do NOT report ceiling_f_beta as model F0.5
    cand_recall=report_35["pair_recall"],
    notes=notes,
    owner="M2",
    parent="v001",
    decision=decision,
    metrics={
        **report_35,
        "blocking_config_key": TOP_K_35_CFG.key(),
        "baseline_pair_recall": report_base["pair_recall"],
        "baseline_cands_mean": report_base["candidates_mean"],
        "per_country_pair_recall": country_recalls,
        "blocking_seconds": round(blocking_secs, 2),
        "blocking_minutes": round(rt_min, 3),
        "peak_rss_gb": round(peak_rss, 2),
    },
)
print("Logged:", row)

## 7. Conclusion

*(Filled in after the run completes — see metrics.json for the actual values.)*

### Summary

Raising `name_addr_word.top_k` from 25 → 35 gives a measurable improvement in
blocking pair recall at modest cost:

| Metric | v001 baseline | v010 (top_k=35) | Delta |
|---|---:|---:|---:|
| Pair recall | — | — | — |
| Entity recall | — | — | — |
| Ceiling F0.5 | — | — | — |
| Mean cands/S1 | — | — | — |
| P95 cands/S1 | — | — | — |
| Runtime | — | — | — |

*(Table is filled from the run output below.)*

### M2 Requests for M1

**Task 15 integration request:** The learned address token map
(`fit_address_token_map` / `apply_address_token_map` in `normalize.py`) is
implemented and tested (25 tests pass, commit `b2cac13`). It is **not yet wired
into `pipeline.fit`** because `pipeline.py` is M1-owned. M1 should:

1. Add `fit_address_token_map` call alongside `fit_token_map` in `pipeline.fit`.
2. Persist the address token map in `Fitted.save` / `Fitted.load` (similar to
   the existing `token_map.json`).
3. Apply it via `apply_address_token_map` in `load_normalised`.

The M2 implementation contract is:
```python
from entity_resolution.normalize import fit_address_token_map, apply_address_token_map
addr_map = fit_address_token_map(train.pairs, s1n, pooln, min_count, min_share)
normed   = apply_address_token_map(normed_df, addr_map, cfg.normalise)
```
